In [22]:
from typing import TypedDict, Annotated
import operator
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
import os

load_dotenv()

True

In [26]:
model = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"))

class Evaluation(BaseModel):
    feedback: str = Field(description="Short feedback on this aspect")
    score: int = Field(ge=0, le=10, description="Score from 0 to 10")

evaluator = model.with_structured_output(Evaluation)

In [11]:
class EssayState(TypedDict):
    essay: str
    correctness_feedback: str
    language_feedback: str
    word_choice_feedback: str
    scores: Annotated[list[int], operator.add]
    final_feedback: str
    average_score: float

In [12]:
def evalute_correctness(state: EssayState):
    prompt = ("Evaluate the correctness of this essay: factual accuracy, logic, "
              f"and whether the arguments hold up.\n\n{state['essay']}")
    r = evaluator.invoke(prompt)
    return {"correctness_feedback": r.feedback, "scores": [r.score]}

In [13]:
def evaluate_language(state: EssayState):
    prompt = ("Evaluate the language quality of this essay: grammar, spelling, "
              f"punctuation, sentence structure and flow.\n\n{state['essay']}")
    r = evaluator.invoke(prompt)
    return {"language_feedback": r.feedback, "scores": [r.score]}

In [14]:

def evaluate_word_choice(state: EssayState):
    prompt = ("Evaluate the word selection in this essay: vocabulary, precision, "
              f"tone and repetition.\n\n{state['essay']}")
    r = evaluator.invoke(prompt)
    return {"word_choice_feedback": r.feedback, "scores": [r.score]}

In [15]:
def final_evaluation(state: EssayState):
    prompt = (
        "Write a concise overall summary of this essay evaluation.\n\n"
        f"Correctness: {state['correctness_feedback']}\n"
        f"Language: {state['language_feedback']}\n"
        f"Word choice: {state['word_choice_feedback']}"
    )
    summary = model.invoke(prompt).content
    avg = sum(state["scores"]) / len(state["scores"])
    return {"final_feedback": summary, "average_score": avg}

In [24]:
graph = StateGraph(EssayState)

graph.add_node("evalute_correctness", evalute_correctness)
graph.add_node("evaluate_language", evaluate_language)
graph.add_node("evaluate_word_choice", evaluate_word_choice)
graph.add_node("final_evaluation", final_evaluation)

# fan-out: all three start together
graph.add_edge(START, "evalute_correctness")
graph.add_edge(START, "evaluate_language")
graph.add_edge(START, "evaluate_word_choice")

# fan-in: final node waits for all three
graph.add_edge("evalute_correctness", "final_evaluation")
graph.add_edge("evaluate_language", "final_evaluation")
graph.add_edge("evaluate_word_choice", "final_evaluation")

graph.add_edge("final_evaluation", END)

workflow = graph.compile()

In [30]:
essay = """Human life is a precious journey filled with learning, growth, and new experiences, It brings moments of joy and success alongside challenges that test our strength.
            Our relationships with family, friends, and society give life deeper meaning, Kindness, empathy, and helping others help us make the world a better place, We should cherish every day, stay grateful, and use our talents wisely
            ...."""

result = workflow.invoke({"essay": essay})
print(result["final_feedback"])
print("Average score:", result["average_score"])

The essay presents a positive view on human life, recognizing both its challenges and joys, but is hampered by grammatical issues, particularly in punctuation and sentence structure, which detract from clarity. While the argument emphasizing the value of relationships and personal growth is logical, it would be strengthened by more detailed examples. Additionally, issues with punctuation lead to run-on sentences and impede the flow between ideas. The vocabulary is generally effective and optimistic, though some repetition of themes occurs. A focus on grammar, punctuation, and coherence would significantly improve the essay's overall impact.
Average score: 5.666666666666667
